In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip /content/drive/MyDrive/ReviewsMidExcludedEqualSamplesSplit.zip -d /content/data

In [ ]:
import numpy as np

import tensorflow as tf
from tensorflow.keras import utils

In [ ]:
BATCH_SIZE = 32
SEED = 42
# VOCAB_SIZE = 5000

In [ ]:
train_data_directory = '/content/data/ReviewsMidExcludedEqualSamplesSplit/train'

In [ ]:
raw_training_dataset = utils.text_dataset_from_directory(
    train_data_directory,
    batch_size=BATCH_SIZE,
    seed=SEED,
    validation_split=0.2,
    subset='training')

In [ ]:
for i, label in enumerate(raw_training_dataset.class_names):
  print("Label", i, "corresponds to", label)

In [ ]:
raw_validation_dataset = utils.text_dataset_from_directory(
    train_data_directory,
    batch_size=BATCH_SIZE,
    seed=SEED,
    validation_split=0.2,
    subset='validation'
)

In [ ]:
test_data_directory = '/content/data/ReviewsMidExcludedEqualSamplesSplit/test'
raw_testing_dataset = utils.text_dataset_from_directory(
    test_data_directory,
    batch_size=BATCH_SIZE
)

In [ ]:
raw_training_dataset_optimized = raw_training_dataset.cache().prefetch(buffer_size=tf.data.AUTOTUNE)
raw_validation_dataset_optimized = raw_validation_dataset.cache().prefetch(buffer_size=tf.data.AUTOTUNE)
raw_testing_dataset_optimized = raw_testing_dataset.prefetch(buffer_size=tf.data.AUTOTUNE)

In [ ]:
from tensorflow.keras.layers import TextVectorization

In [ ]:
# encoder = TextVectorization(max_tokens=VOCAB_SIZE)
encoder = TextVectorization()
encoder.adapt(raw_training_dataset_optimized.map(lambda text, label: text))

In [ ]:
vocab = np.array(encoder.get_vocabulary())
vocab[1500:]

In [ ]:
!pip install keras-tuner

In [ ]:
import keras_tuner as kt

In [ ]:
def model_builder(hp):
    model = tf.keras.Sequential()
    model.add(encoder)

    hp_output_dim = hp.Int('output_dim', 16, 160, step=16)
    model.add(tf.keras.layers.Embedding(
        input_dim=len(encoder.get_vocabulary()),
        output_dim=hp_output_dim,
        # Use masking to handle the variable sequence lengths
        mask_zero=True))

    hp_LSTM_units = hp.Int('LSTM_units', 16, 160, step=16)
    model.add(tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(hp_LSTM_units)))

    hp_Dense_units = hp.Int('Dense_units', 16, 160, step=16)
    hp_Dense_activations_first = hp.Choice('Dense_activation_first', ['relu', 'sigmoid', 'softmax', 'tanh', 'linear'])
    model.add(tf.keras.layers.Dense(hp_Dense_units, activation=hp_Dense_activations_first))

    hp_Dense_activations_second = hp.Choice('Dense_activation_second', ['relu', 'sigmoid', 'softmax', 'tanh', 'linear'])
    model.add(tf.keras.layers.Dense(1, activation=hp_Dense_activations_second))

    hp_learning_rate = hp.Choice('learning_rate', [1e-5, 2e-5, 5e-5, 1e-4, 2e-4, 5e-4, 1e-3, 2e-3, 5e-3])
    model.compile(loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
                  optimizer=tf.keras.optimizers.Adam(hp_learning_rate),
                  metrics=['accuracy'])
    return model

In [ ]:
tuner = kt.Hyperband(model_builder,
                     objective='val_accuracy',
                     max_epochs=10)

In [ ]:
stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)

In [ ]:
tuner.search(raw_training_dataset_optimized, epochs=10,
             validation_data=raw_validation_dataset_optimized,
             validation_steps=30)

In [ ]:
best_hps = tuner.get_best_hyperparameters()[0]
print('output_dim:', best_hps.get('output_dim'))
print('LSTM_units:', best_hps.get('LSTM_units'))
print('Dense_units:', best_hps.get('Dense_units'))
print('Dense_activation_first:', best_hps.get('Dense_activation_first'))
print('Dense_activation_second:', best_hps.get('Dense_activation_second'))
print('learning_rate:', best_hps.get('learning_rate'))